# Stage 2: Instruction Fine-Tuning

**Project:** IT Helpdesk AI Assistant (domain-specific fine-tuning with Unsloth)

**Goal:** Teach the model to answer IT Helpdesk questions directly, continuing from the Stage 1 non-instruction adapter.

> **Run this notebook on a GPU runtime** (Google Colab free T4, or Kaggle GPU).

Steps covered:
1. Install required libraries
2. Load tokenizer
3. Load model (continuing from Stage 1 adapter)
4. Format the instruction dataset
5. Apply LoRA/QLoRA
6. Train the model
7. Save adapter/model
8. Run inference after training

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## 1. Load tokenizer and base/adapter model

We continue from the Stage 1 non-instruction adapter saved at `models/non_instruction_adapter`. If that adapter is not available (e.g. running standalone), fall back to the base model.

In [ ]:
from unsloth import FastLanguageModel
import torch, os

MAX_SEQ_LENGTH = 1024
STAGE1_ADAPTER = "models/non_instruction_adapter"
BASE_MODEL = "unsloth/tinyllama-bnb-4bit"  # TinyLlama-1.1B

model_name = STAGE1_ADAPTER if os.path.isdir(STAGE1_ADAPTER) else BASE_MODEL

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

## 2. Load and format the instruction dataset

`data/instruction_dataset.jsonl` contains `{"instruction": ..., "response": ...}` records. We wrap them in a simple chat-style prompt template.

In [ ]:
import json
from datasets import Dataset

PROMPT_TEMPLATE = """Below is an IT Helpdesk support request. Write a helpful, professional response.

### Request:
{instruction}

### Response:
{response}"""

records = []
with open("../data/instruction_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} instruction examples")

EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    text = PROMPT_TEMPLATE.format(instruction=example["instruction"], response=example["response"]) + EOS_TOKEN
    return {"text": text}

dataset = Dataset.from_list(records).map(format_example)
print(dataset[0]["text"][:400])

## 3. Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Train the model (SFT)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_instruction",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

## 5. Save adapter/model

In [ ]:
SAVE_DIR = "models/sft_adapter"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved instruction-tuned (SFT) adapter to {SAVE_DIR}")

## 6. Run inference after training

Use the same 10 evaluation questions from `reports/base_model_evaluation.md` to build `reports/sft_model_comparison.md`.

In [ ]:
import json
import sys

sys.path.append("../src")
from eval_questions import EVAL_QUESTIONS

FastLanguageModel.for_inference(model)

sft_answers = []
for q in EVAL_QUESTIONS:
    prompt = PROMPT_TEMPLATE.format(instruction=q, response="")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded.split("### Response:")[-1].strip()
    sft_answers.append({"question": q, "answer": answer})
    print("Q:", q)
    print("A:", answer)
    print("-" * 80)

with open("../reports/sft_model_answers.json", "w", encoding="utf-8") as f:
    json.dump(sft_answers, f, indent=2)
print("Saved SFT answers to reports/sft_model_answers.json")